# This notebook is for unit tests on tools and models

In [3]:
import pandas as pd
import time
from config import TRADE_LIST
from src.utils.CalFactorFramework import FactorCalculator, FactorRegistry

## test CalFactorFramework.py

In [2]:
from src.models.factors.rv import compute as rv_daily

In [2]:
calc = FactorCalculator()
registry = FactorRegistry(calc, "./register/factor_registry.json")

2025-10-15 23:40:00 [INFO] FactorRegistry: Loaded metadata for 2 factors from register\factor_registry.json
2025-10-15 23:40:00 [INFO] FactorRegistry: Loaded metadata for 2 factors from register\factor_registry.json
2025-10-15 23:40:00 [WARNING] FactorRegistry: Failed to load logs from logs\factor_update_log.json: Extra data: line 1 column 3 (char 2)


Loaded metadata for 2 factors from register\factor_registry.json


In [4]:
register_config = {'name': 'rv_daily', 'factor_func': rv_daily, 'frequency': 'min',
                   'fields': ["symbol", "timestamp", "Close"],
                   'type_': 'alpha', 'category': 'volatility', 'description': 'test'}

registry.register(**register_config) # 使用默认数据注册
# registry.register(**register_config, test=1) register_config之后添加的是动态参数

2025-10-10 21:46:30 [INFO] FactorRegistry: Factor 'rv_daily' registered: src.utils.factors.rv_daily.compute


In [5]:
registry.list_factors()

{'rv_daily': {'description': 'test',
  'frequency': 'min',
  'fields': ['symbol', 'timestamp', 'Close'],
  'created_time': '2025-10-10T21:46:30.950572',
  'updated_time': '2025-10-10T21:46:30.950572',
  'type': 'alpha',
  'category': 'volatility',
  'has_function': True}}

log有输出，说明已经被注册

### min_data 的两种计算模式：
1. 直接计算，is_batch参数设为False，直接读取所有数据（如果是指定了dates就是读取所有dates，不指定就读取目录下所有），可选择是否并行（parallel_batch_size参数决定每个并行进程处理多少日的数据）
2. 分批伪增量运算，为避免memory overflow，每次只读取一定批次的数据（即为batch_size的取值，可以理解为串行的多次直接运算），parallel_batch_size，n_jobs与batch_size需要互相配合。

ps: 在parallel_batch_size小于60时，读取分钟数据为正常循环读取，大于60采取多线程（每个进程内部的多线程以提高效率，为了避免过分的资源争抢，设定为固定8个线程），所以parallel_batch_size，n_jobs与batch_size需要互相配合。

In [8]:
# test direct calculation
# 没有分批次增量计算，没有使用多进程
start_single_process = time.time()
registry.calculate(
        'rv_daily',
        is_batch=True,
        is_parallel=False,
        symbols=TRADE_LIST,
        price_col='Close'  # 这里的是动态参数，根据因子计算函数要求给
    )
end_single_process = time.time()
print(f"single-process takes {end_single_process - start_single_process} seconds")
res = registry.get_factor_data('rv_daily')
print(res.head())
print("===============================")
print(res.shape)

Processing Date Batch 1/2: 20250101 to 20250629
Processing Date Batch 2/2: 20250630 to 20250910
single-process takes 255.25425601005554 seconds


FactorRegistryError: Calculator does not support 'get_factor_data' method

In [9]:
# test multi-process
start_multi_process = time.time()
registry.calculate(
        'rv_daily',
        is_batch=True,
        is_parallel=True,
        symbols=TRADE_LIST,
        batch_size=120, # 每个增量的容量
        parallel_batch_size=20, # 每个进程要处理多少天数据
        n_jobs=6
    )  # 增量是处于内存考虑，记一次同时读取多少填数据，如果内存够大，batch_size直接取天数也可以尝试
end_multi_process = time.time()
print(f"multi-process takes {end_multi_process - start_multi_process} seconds")
res1 = registry.get_factor_data('rv_daily')
print(res1.head())
print("===============================")
print(res1.shape)

Processing Date Batch 1/7: 20250101 to 20250209
Processing Date Batch 2/7: 20250210 to 20250321
Processing Date Batch 3/7: 20250322 to 20250430
Processing Date Batch 4/7: 20250501 to 20250609
Processing Date Batch 5/7: 20250610 to 20250719
Processing Date Batch 6/7: 20250720 to 20250828
Processing Date Batch 7/7: 20250829 to 20250910
multi-process takes 255.25425601005554 seconds


FactorRegistryError: Calculator does not support 'get_factor_data' method

In [ ]:
# 理论上更好的参数组合
start_multi_process = time.time()
registry.calculate(
        'rv_daily',
        is_batch=False,
        is_parallel=True,
        symbols=TRADE_LIST,
        batch_size=400,
        parallel_batch_size=80,
        n_jobs=5
    )
end_multi_process = time.time()
print(f"multi-process takes {end_multi_process - start_multi_process} seconds")
res1 = registry.get_factor_data('rv_daily')
print(res1.head())
print("===============================")
print(res1.shape)


## test BacktestTool.py

In [3]:
from src.utils.BacktestTools import SingleAlphaFactorBacktester, BacktestContext

In [4]:
test = registry.get_factor_data('rv_daily').drop_duplicates(subset=['symbol', 'trade_date', 'rv_daily'], keep='first')
test.timestamp = pd.to_datetime(test.timestamp, unit='ms')

In [5]:
close = pd.read_parquet('./data/daily_data/all_data.parquet')[['symbol', 'timestamp', 'Close']]

In [6]:
test_contest = BacktestContext(factor_df=test.set_index(['symbol', 'timestamp'], drop=False), factor_name='rv_daily', price_data=close.set_index(['symbol', 'timestamp'], drop=False), rebalance_freq='1D', price_col='Close')

In [8]:
test_backtest = SingleAlphaFactorBacktester(10)

In [10]:
results = test_backtest.run(test_contest)

In [ ]:
test_backtest.plot_summary(results)

In [4]:
from pathlib import Path
import sys
import types

In [2]:
from pathlib import Path
import sys, types

# shim nbclient（与之前相同）
nbclient = types.ModuleType("nbclient")
nbclient_client = types.ModuleType("nbclient.client")
nbclient_client.timestamp = None
nbclient.client = nbclient_client
sys.modules["nbclient"] = nbclient
sys.modules["nbclient.client"] = nbclient_client

sys.path.insert(0, str(Path('.').resolve()))

from src.utils.CalFactorFramework import FactorCalculator, FactorRegistry
from src.models.factors.trend_regression_hourly import register

calc = FactorCalculator("./data")
registry = FactorRegistry(calc, "./register/factor_registry.json")

if not registry.exists("trend_hour_4h"):
    register(registry, lookahead_hours=4, warmup_hours=1200)

registry.calculate(
    "trend_hour_4h",
    is_batch=False,
    is_parallel=False,
    price_col="Close",
    lookahead_hours=4,
    beta_span=48,      # 可省略，默认也是 48
    warmup_hours=1200,   # 与 8h 一样，跳过前 1200 小时
)

print("Saved", Path("data/factors/alpha/factor_trend_hour_4h.parquet").resolve())

2025-10-24 14:18:09 [INFO] FactorRegistry: Loaded metadata for 27 factors from register/factor_registry.json
2025-10-24 14:18:09 [WARNING] FactorRegistry: Failed to load logs from logs/factor_update_log.json: Extra data: line 2 column 1 (char 147)


Loaded metadata for 27 factors from register/factor_registry.json
Calculating compute...
Saved /home/mfin7037_best_students/multi_branch_gru/aveniur-hku_comp/data/factors/alpha/factor_trend_hour_4h.parquet


In [ ]:
from pathlib import Path
import sys, types

nbclient = types.ModuleType("nbclient")
nbclient_client = types.ModuleType("nbclient.client")
nbclient_client.timestamp = None
nbclient.client = nbclient_client
sys.modules["nbclient"] = nbclient
sys.modules["nbclient.client"] = nbclient_client

sys.path.insert(0, str(Path('.').resolve()))

from src.utils.CalFactorFramework import FactorCalculator, FactorRegistry
from src.models.factors.trend_regression_hourly import register

calc = FactorCalculator("./data")
registry = FactorRegistry(calc, "./register/factor_registry.json")

if not registry.exists("trend_hour_8h"):
    register(registry, lookahead_hours=8, warmup_hours=1200)

registry.calculate(
    "trend_hour_8h",
    is_batch=False,
    is_parallel=False,
    price_col="Close",
    lookahead_hours=8,
    beta_span=96,
    warmup_hours=1200,
)
print("Saved", Path("data/factors/alpha/factor_trend_hour_8h.parquet").resolve())

2025-10-24 13:08:30 [INFO] FactorRegistry: Loaded metadata for 27 factors from register/factor_registry.json
2025-10-24 13:08:30 [WARNING] FactorRegistry: Failed to load logs from logs/factor_update_log.json: Extra data: line 2 column 1 (char 147)


Loaded metadata for 27 factors from register/factor_registry.json
Calculating compute...
Saved /home/mfin7037_best_students/multi_branch_gru/aveniur-hku_comp/data/factors/alpha/factor_trend_hour_8h.parquet
